# Workshop 4 · Autoencoders y VAE en Colab
Controlador de `src/train_ae.py`: el entrenamiento, validación, selección y reanudación viven en Python. El notebook no contiene arquitecturas ni pérdidas.

1. En Colab selecciona **Entorno de ejecución → Cambiar tipo → GPU**.
2. Exporta el proyecto a ZIP como explica el README; inclúyelo en Drive o súbelo aquí. El ZIP contiene Workshop_4 y únicamente vocab/splits de Workshop_1; los Parquet permanecen en Drive.
3. Ajusta rutas y parámetros, revisa el dry run y habilita `RUN_TRAINING` para ejecutar.
4. Los resultados se escriben directamente en Drive. Una desconexión conserva los checkpoints ya publicados; reanudar continúa desde la última época completada, no desde el minibatch interrumpido.

**Esta copia se entrega sin ejecutar experimentos.** El test no se usa para seleccionar modelos. Los experimentos completos son de 30–50 épocas; ejecuta primero un smoke test mediante el CLI según README.


**Actualizaci?n auditada:** mientras siga activo ae_z32 en Colab, no ejecutar estas celdas ni reemplazar sus archivos. Esperar a que termine; despu?s usar una carpeta de extracci?n y salida nuevas. El informe AUDIT_REPORT.md distingue hip?tesis de causas confirmadas y describe el benchmark T4.


In [ ]:
from google.colab import drive, files
drive.mount("/content/drive")
from pathlib import Path
import sys, subprocess, zipfile


In [ ]:
# @title Rutas de proyecto, datos y resultados
ZIP_SOURCE = "Drive" # @param ["Drive", "Upload"]
PROJECT_ZIP = "/content/drive/MyDrive/Workshop_4/workshop4_colab.zip" # @param {type:"string"}
EXTRACT_DIR = "/content/workshop4_project" # @param {type:"string"}
DRIVE_PROCESSED_DIR = "/content/drive/MyDrive/Workshop_4/data/processed" # @param {type:"string"}
OUTPUT_DIR = "/content/drive/MyDrive/Workshop_4/runs/workshop4" # @param {type:"string"}
COPY_TRAIN_VAL_TO_LOCAL = True # @param {type:"boolean"}
LOCAL_PROCESSED_DIR = "/content/workshop4_data/processed" # @param {type:"string"}


In [ ]:
if ZIP_SOURCE == "Upload":
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Sube exactamente un ZIP del proyecto.")
    PROJECT_ZIP = next(iter(uploaded))
zip_path = Path(PROJECT_ZIP).expanduser().resolve()
if not zip_path.is_file():
    raise FileNotFoundError(zip_path)
# Cargar el helper directamente del ZIP propio antes de extraer el proyecto.
with zipfile.ZipFile(zip_path) as archive:
    helpers = [name for name in archive.namelist()
               if name.endswith("Workshop_4/src/colab_utils.py")]
    if len(helpers) != 1:
        raise ValueError("El ZIP debe contener un único Workshop_4/src/colab_utils.py")
    zip_import_root = helpers[0].removesuffix("src/colab_utils.py")
sys.path.insert(0, str(zip_path) + "/" + zip_import_root)
from src.colab_utils import (
    extract_project_zip, stage_processed_parquet, runtime_info,
    build_training_command, plot_history,
)
WORKSHOP_DIR = extract_project_zip(zip_path, EXTRACT_DIR)
PROCESSED_DIR = stage_processed_parquet(
    DRIVE_PROCESSED_DIR,
    LOCAL_PROCESSED_DIR if COPY_TRAIN_VAL_TO_LOCAL else None,
)
print("Código:", WORKSHOP_DIR)
print("Train/val:", PROCESSED_DIR)
print("Resultados persistentes:", OUTPUT_DIR)


In [ ]:
# Dependencias auxiliares solo si faltan; no sustituye el PyTorch de Colab.
# experiments.yaml usa JSON compatible con YAML; PyYAML solo es opcional para sintaxis YAML plena.
import importlib.util
packages = {"pyarrow": "pyarrow>=14,<27", "matplotlib": "matplotlib>=3.7,<4"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
info = runtime_info()
print(info)
if not info["cuda_available"]:
    print("GPU no disponible: activa un runtime GPU o selecciona CPU explícitamente abajo.")


In [ ]:
# @title Panel editable de experimentos
EXPERIMENT_SET = "Recommended (5)" # @param ["Recommended (5)", "Selected", "All including optional (12)"]
EXPERIMENT_NAMES = "ae_z32,vae_z32_b1" # @param {type:"string"}
EPOCHS = 30 # @param {type:"integer"}
BATCH_SIZE = 512 # @param {type:"integer"}
LEARNING_RATE = 0.001 # @param {type:"number"}
SEED = 42 # @param {type:"integer"}
DEVICE = "cuda" # @param ["cuda", "cpu", "auto"]
# 0 mantiene las dimensiones 32/64/128 de experiments.yaml.
LATENT_DIM_OVERRIDE = 0 # @param {type:"integer"}
# Vacío conserva beta por experimento: 0.5, 1.0 o 4.0.
BETA_OVERRIDE = "" # @param {type:"string"}
RESUME = False # @param {type:"boolean"}
RUN_TRAINING = False # @param {type:"boolean"}
# Opciones de transporte: medir antes de adoptarlas en un experimento nuevo.
TENSOR_LOADER = False # @param {type:"boolean"}
PREFETCH_BATCHES = 0 # @param {type:"integer"}
PIN_MEMORY = False # @param {type:"boolean"}
PROFILE_BATCHES = 0 # @param {type:"integer"}
GRADIENT_LOG_EVERY = 1 # @param {type:"integer"}

selected = None if EXPERIMENT_SET != "Selected" else [name.strip() for name in EXPERIMENT_NAMES.split(",") if name.strip()]
options = dict(
    workshop_dir=WORKSHOP_DIR, output_dir=OUTPUT_DIR, processed_dir=PROCESSED_DIR,
    experiments=selected, run_all=EXPERIMENT_SET == "All including optional (12)",
    recommended=EXPERIMENT_SET == "Recommended (5)", device=DEVICE,
    tensor_loader=TENSOR_LOADER, prefetch_batches=PREFETCH_BATCHES, pin_memory=PIN_MEMORY,
    profile_batches=PROFILE_BATCHES, gradient_log_every=GRADIENT_LOG_EVERY,
    epochs=EPOCHS, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, seed=SEED,
    latent_dim=LATENT_DIM_OVERRIDE or None,
    beta=float(BETA_OVERRIDE) if BETA_OVERRIDE.strip() else None,
    resume=RESUME,
)


Los nombres disponibles son `ae_z32`, `ae_z64`, `ae_z128` y `vae_z{32,64,128}_b{0p5,1,4}`.
`EXPERIMENT_SET="Recommended (5)"` selecciona AE z32/z64 y VAE z32 con beta 0.5/1/4. `All including optional (12)` requiere elegir deliberadamente la grilla completa. Los overrides se aplican a todos los seleccionados; conserva 0/vacío para recorrer dimensiones/betas del YAML.

Los nombres de las carpetas conservan el nombre del YAML incluso si aplicas overrides; `metadata.json` registra los hiperpar?metros efectivos.

Usa un nuevo `OUTPUT_DIR` si cambias hiperparámetros de un experimento ya iniciado. Para continuar sin cambiar su configuración, activa `RESUME`. El entrenador valida el contrato de datos y los hiperparámetros del checkpoint.

En cada carpeta de experimento se guardan `metadata.json`, `history.csv`, `checkpoints/best.pt` y `checkpoints/last.pt`. `best.pt` se selecciona con validación, nunca con test. `last.pt` conserva el estado de Adam y los generadores aleatorios.


In [ ]:
# Revisa los experimentos efectivos sin entrenar ni crear resultados.
dry_command = build_training_command(**options, dry_run=True)
subprocess.run(dry_command, cwd=WORKSHOP_DIR, check=True)


In [ ]:
if RUN_TRAINING:
    command = build_training_command(**options)
    print("Ejecutando:", command)
    subprocess.run(command, cwd=WORKSHOP_DIR, check=True)
else:
    print("Entrenamiento deshabilitado. Revisa el dry run y activa RUN_TRAINING para comenzar.")


In [ ]:
# Lee únicamente los resultados existentes; guarda training_curves.png en Drive.
from IPython.display import display
figures = plot_history(OUTPUT_DIR, experiments=selected)
for figure in figures:
    display(figure)


Al terminar, descarga desde Drive las carpetas de los experimentos que quieras conservar. Los checkpoints son portables CPU/CUDA y contienen el vocabulario, el contrato de datos y los hiperparámetros efectivos.

El raw y test permanecen intactos en Drive. El protocolo de anomalías (entrenar solo con datos normales, umbral en validación) se ejecutará como una tarea separada; los runs de representación no sustituyen ese requisito del PDF. Ninguna métrica de test se calcula en este notebook.
